# 精細調校 Open AI 模型

此筆記本基於 Open AI 提供的[精細調校](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)文件中的最新指引。

> **注意：** 此筆記本的示例輸出是以 `gpt-3.5-turbo` 為基準生成，但該模型現已在 Microsoft Foundry 的 Azure OpenAI 中停止推理和精細調校（並被 OpenAI 官方廢止）。以下的說明和概念仍然準確，但如果你今天要開始新的精細調校任務，請選擇目前支援的模型，例如 `gpt-4o-mini` 或 `gpt-4.1-mini`。請參閱[精細調校模型清單](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)以了解目前可精細調校的模型集。

精細調校透過使用與特定用例或場景相關的額外數據和上下文，對基底模型進行重新訓練，從而提升模型在應用中的表現。請注意，像是 _少量學習_ 和 _檢索增強生成_ 這類提示工程技術，允許你用相關資料強化預設提示以提升質量，但這些方法受到目標基底模型最大字元窗口大小的限制。

透過精細調校，我們實際上是在使用需求數據重新訓練模型（使我們可以使用比最大字元窗口更多的示例）— 並部署一個 _自訂_ 版本的模型，該模型在推理時不再需要提供示例。這不僅提升了提示設計的有效性（我們在使用字元窗口上有更多彈性），也可能改善成本（因為推理時送往模型的字元數量減少）。

精細調校包含 4 個步驟：
1. 準備訓練數據並上傳。
1. 執行訓練作業以獲得精細調校模型。
1. 評估精細調校模型並針對品質進行迭代。
1. 確定滿意後部署精細調校模型進行推理。

請注意，並非所有基底模型都支援精細調校 - 請參閱[OpenAI文件](https://platform.openai.com/docs/guides/fine-tuning/what-models-can-be-fine-tuned?WT.mc_id=academic-105485-koreyst)了解最新資訊。你也可以對先前精細調校過的模型再次進行精細調校。本教學中，我們將以 `gpt-35-turbo` 作為精細調校的目標基底模型（參見上方說明以選擇目前支援的替代方案）。

---


### 第 1.1 步：準備您的數據集

我們來建立一個聊天機械人，幫助您通過回答關於元素的問題並用打油詩來解說周期表。在_此_簡易教程中，我們將只創建一個數據集來訓練模型，內含幾個展示期望數據格式的示例回應。在真實的應用場景中，您需要創建一個包含更多示例的數據集。如果存在，您也可以使用開源數據集（針對您的應用領域），並重新格式化以便用於微調。

由於我們專注於 `gpt-35-turbo` 並尋求單輪回答（聊天完成），我們可以使用[此建議格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)建立示例，符合 OpenAI 聊天完成的要求。如果您預期是多輪對話內容，將使用包含 `weight` 參數的[多輪示例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，以標示哪些訊息應該在微調過程中使用（或不使用）。

我們在此教程中將使用較簡單的單輪格式。數據為 [jsonl 格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行一條紀錄，以 JSON 格式物件表示。以下片段示範 2 條紀錄作為範例—完整示例集（10 個示例）請參閱 [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl)，將用於我們的微調教程。<strong>注意：</strong>每條紀錄_必須_定義在單行中（不可分多行，就像格式化的 JSON 文件那樣）。

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在真實應用中，為達較好效果您需要更多示例—權衡點是回應質量與微調的時間／成本。我們使用小型示例集，是為了能快速完成微調，說明過程。更多複雜的微調教程請參考[此 OpenAI Cookbook 範例](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)。


---

### 第1.2步 上載你的數據集

使用 Files API [如此處所述](https://platform.openai.com/docs/guides/fine-tuning/upload-a-training-file) 上載數據。請注意，要運行此代碼，您必須先完成以下步驟：
 - 安裝 `openai` Python 套件（確保使用版本 >=0.28.0 以支援最新功能）
 - 將 `OPENAI_API_KEY` 環境變量設定為您的 OpenAI API 金鑰
如欲了解更多，請參閱本課程提供的 [設定指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)。

現在，運行代碼以從您的本地 JSONL 文件創建待上載文件。


In [24]:
from openai import OpenAI
client = OpenAI()

ft_file = client.files.create(
  file=open("./training-data.jsonl", "rb"),
  purpose="fine-tune"
)

print(ft_file)
print("Training File ID: " + ft_file.id)

FileObject(id='file-JdAJcagdOTG6ACNlFWzuzmyV', bytes=4021, created_at=1715566183, filename='training-data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)
Training File ID: file-JdAJcagdOTG6ACNlFWzuzmyV


---

### 步驟 2.1：使用 SDK 建立微調工作


In [25]:
from openai import OpenAI
client = OpenAI()

ft_filejob = client.fine_tuning.jobs.create(
  training_file=ft_file.id, 
  model="gpt-3.5-turbo"
)

print(ft_filejob)
print("Fine-tuning Job ID: " + ft_filejob.id)

FineTuningJob(id='ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', created_at=1715566184, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-EZ6ag0n0S6Zm8eV9BSWKmE6l', result_files=[], seed=830529052, status='validating_files', trained_tokens=None, training_file='file-JdAJcagdOTG6ACNlFWzuzmyV', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)
Fine-tuning Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh


---

### 步驟 2.2：檢查工作的狀態

以下是您可以使用 `client.fine_tuning.jobs` API 做的一些事情：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最新的 n 個微調工作
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 獲取特定微調工作的詳細資訊
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消微調工作
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<b>)` - 列出該工作的最多 n 個事件
- `client.fine_tuning.jobs.create(model="gpt-35-turbo", training_file="your-training-file.jsonl", ...)`

流程的第一步是 _驗證訓練文件_，以確保資料格式正確。


In [26]:
from openai import OpenAI
client = OpenAI()

# List 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(ft_filejob.id)

# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_filejob.id, limit=10)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-GkWiDgZmOsuv4q5cSTEGscY6', created_at=1715566184, level='info', message='Validating training file: file-JdAJcagdOTG6ACNlFWzuzmyV', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-3899xdVTO3LN7Q7LkKLMJUnb', created_at=1715566184, level='info', message='Created fine-tuning job: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=False)

In [30]:
# Once the training data is validated
# Track the job status to see if it is running and when it is complete
from openai import OpenAI
client = OpenAI()

response = client.fine_tuning.jobs.retrieve(ft_filejob.id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh
Status: running
Trained Tokens: None


---

### 步驟 2.3：追蹤事件以監控進度


In [44]:
# You can also track progress in a more granular way by checking for events
# Refresh this code till you get the `The job has successfully completed` message
response = client.fine_tuning.jobs.list_events(ft_filejob.id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 85/100: training loss=0.14
Step 86/100: training loss=0.00
Step 87/100: training loss=0.00
Step 88/100: training loss=0.07
Step 89/100: training loss=0.00
Step 90/100: training loss=0.00
Step 91/100: training loss=0.00
Step 92/100: training loss=0.00
Step 93/100: training loss=0.00
Step 94/100: training loss=0.00
Step 95/100: training loss=0.08
Step 96/100: training loss=0.05
Step 97/100: training loss=0.00
Step 98/100: training loss=0.00
Step 99/100: training loss=0.00
Step 100/100: training loss=0.00
Checkpoint created at step 80 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyyF2:ckpt-step-80
Checkpoint created at step 90 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyzhK:ckpt-step-90
New fine-tuned model created: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz
The job has successfully completed


### 步驟 2.4：在 OpenAI 控制台查看狀態


你亦可以通過訪問 OpenAI 網站並瀏覽平台的 _Fine-tuning_ 部分來查看狀態。這會顯示當前工作的狀態，並讓你追蹤先前工作執行的歷史記錄。在這張截圖中，你可以看到先前的執行失敗了，而第二次執行成功了。作為背景說明，這是因為第一次執行時使用了格式錯誤的 JSON 文件記錄 - 修正後，第二次執行成功完成，並使模型可用。

![Fine-tuning job status](../../../../../translated_images/zh-HK/fine-tuned-model-status.563271727bf7bfba.webp)


您亦可以在視覺化儀表板中向下捲動，查看狀態訊息及指標，如圖所示：

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-HK/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-HK/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 步驟 3.1：在程式碼中取回 ID 並測試微調模型


In [46]:
# Retrieve the identity of the fine-tuned model once ready
response = client.fine_tuning.jobs.retrieve(ft_filejob.id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)

Fine-tuned Model ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz


In [ ]:
# You can then use that model to generate completions from the SDK as shown
# Or you can load that model into the OpenAI Playground (in the UI) to validate it from there.
from openai import OpenAI
client = OpenAI()

completion = client.responses.create(
  model=fine_tuned_model_id,
  input=[
    {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Strontium"},
  ],
  store=False,
)
print(completion.output_text)


ChatCompletionMessage(content="Strontium, a metal so bright - It's in fireworks, a dazzling sight - It's in bones, you see - And in tea, it's the key - It's the fortieth, so pure, that's the right", role='assistant', function_call=None, tool_calls=None)


---

### 步驟 3.2：在 Playground 中載入及測試微調模型

你現在可以用兩種方式測試微調模型。首先，你可以到 Playground，並在模型下拉選單中從列表選擇你剛微調好的模型。另一個選項是使用微調面板中顯示的「Playground」按鈕（見上面截圖），它會啟動以下的 _比較_ 視圖，並並排顯示基礎模型與微調模型版本，方便快速評估。

![微調工作狀態](../../../../../translated_images/zh-HK/fine-tuned-playground-compare.56e06f0ad8922016.webp)

只要填寫你訓練資料中使用的系統上下文，並提供你的測試問題。你會注意到雙方都會用相同的上下文和問題來更新。執行比較，你就會看到它們的輸出差異。_注意微調模型如何以你示例中提供的格式回應，而基礎模型則單純遵循系統提示_。

![微調工作狀態](../../../../../translated_images/zh-HK/fine-tuned-playground-launch.5a26495c983c6350.webp)

你會注意到比較還會提供每個模型的 token 數和推論所用時間。**這個例子相對簡單，只是為了展示流程，並不反映真實世界的數據或情況**。你可能會看到兩組範例使用的 token 數相同（系統上下文和用戶提示一致），但微調模型的推論時間較長（因為是自訂模型）。

在真實世界情況中，你不會用這樣的玩具例子，而是會針對真實資料（例如客戶服務用的產品目錄）進行微調，屆時回應的品質差異將會更明顯。在 _那_ 種情況下，要用基礎模型得到相等的回應品質會需要更多定制提示工程，這會增加 token 使用量及可能的推論處理時間。_想要試試看，可以先參考 OpenAI Cookbook 中的微調範例開始_。

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
